# Chapter 22 — Resistance of a Conductor
### *Modern Strain Gage Technology* — Companion Notebook

**At its core, the strain gage is a resistor** — and everything it does follows from one equation, $R = \rho L / A$: stretch the conductor and its length rises while its cross-section shrinks, so its resistance climbs in a way we can predict and measure.

*Companion notebook to Chapter 22 — Resistance of a Conductor.*

This notebook works through three consequences of $R = \rho L / A$ that the chapter develops in prose:

1. **Resistance from material and geometry.** We build a nominal 120 Ω from a realistic Constantan foil element, using the *total* conductor length of the folded grid — the quantity the chapter's "Why the Grid Is Folded" section is really about.
2. **Gage factor.** We reproduce the chapter's worked example, $\Delta R = GF\,\varepsilon\,R$, and split the gage factor into its geometric and piezoresistive parts exactly as in Eq. 22.4.
3. **Lead-wire resistance error.** The same $R = \rho L / A$ that makes the gage work also governs a quieter problem — the resistance of the wires that connect the gage to the instrument. In a two-wire quarter bridge those leads sit in series with the gage, so any drift in their resistance masquerades as strain.

Everything here is deterministic: run the cells top to bottom and the numbers reproduce exactly.

In [ ]:
# Colab setup — uncomment to install dependencies in a fresh runtime:
# !pip install numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 12,
})

# This notebook is fully deterministic — no random numbers are used.

---
## 1 — Resistance from material and geometry

The resistance of a uniform conductor is fixed entirely by its material and its shape (Eq. 22.1 in the chapter):

$$R = \rho\,\frac{L}{A}$$

In plain terms: resistance grows with resistivity and length, and falls as the cross-section widens. To make a usable gage we need a fair amount of resistance — typically **120 Ω** or **350 Ω** — packed into a few millimeters of surface. A single straight 4 mm wire cannot supply it, which is why the grid is *folded*: the conductor runs back and forth so that its **total length** is many times the active grid length. It is that total conductor length, not the few-millimeter footprint, that enters $R = \rho L / A$.

The values below are representative of a **Constantan** foil element: resistivity $\rho \approx 490\ \text{n}\Omega\cdot\text{m}$, rolled-foil thickness $\approx 5\ \mu\text{m}$, and grid-line width $\approx 25\ \mu\text{m}$. With a total folded-conductor length near 30 mm, they land close to the standard 120 Ω package value.

In [ ]:
def resistance(rho, length, area):
    """Resistance of a uniform conductor, R = rho * L / A (ohms).

    Parameters
    ----------
    rho : float
        Electrical resistivity of the conductor material (ohm-metre).
    length : float
        Total conductor length along the current path (metre). For a
        folded foil grid this is the summed length of every pass, not
        the short active grid length.
    area : float
        Cross-sectional area of the conductor (square metre).
    """
    return rho * length / area

# Representative Constantan foil element.
RHO_CONSTANTAN = 490e-9   # ohm-metre  (Constantan, ~55% Cu / 45% Ni)
FOIL_THICKNESS = 5e-6     # m   typical rolled foil (~5 micron)
LINE_WIDTH     = 25e-6    # m   typical grid-line width
GRID_LENGTH    = 0.030    # m   total *folded* conductor length (~30 mm)

area = LINE_WIDTH * FOIL_THICKNESS
R_grid = resistance(RHO_CONSTANTAN, GRID_LENGTH, area)

print(f"Cross-sectional area  A = w*t = {area:.2e} m^2")
print(f"Resistance            R = rho*L/A = {R_grid:.1f} ohm")
print("  -> close to the standard 120 ohm package value")

---
## 2 — Gage factor: geometry plus piezoresistance

The **gage factor** ties resistance change directly to strain (Eq. 22.3):

$$GF = \frac{\Delta R / R}{\varepsilon}$$

The chapter splits it into two physical sources (Eq. 22.4): a purely **geometric** part, $1 + 2\nu$, from the conductor becoming longer and thinner, and a **piezoresistive** part from the strain-dependence of resistivity. For a metal with $\nu \approx 0.3$ the geometry alone contributes about 1.6; the measured $GF \approx 2.0$ of a Constantan gage shows that piezoresistance supplies the remaining ~0.4 — secondary, but not negligible.

Below we reproduce the chapter's worked example — a 120 Ω Constantan gage at 1{,}850 microstrain — and then back out the piezoresistive share of a $GF = 2.05$ gage.

In [ ]:
# --- Chapter worked example: resistance change under load ---
GF = 2.05          # gage factor of the Constantan gage
R_NOMINAL = 120.0  # ohm, nominal gage resistance
strain = 1850e-6   # applied axial strain (1,850 microstrain)

dR = GF * strain * R_NOMINAL
print(f"Delta R = GF * eps * R = {GF} * {strain:.3e} * {R_NOMINAL:.0f} = {dR:.3f} ohm")
print(f"Fractional change      = Delta R / R = {dR / R_NOMINAL * 100:.3f} %")

# --- Splitting the gage factor into geometric and piezoresistive parts (Eq. 22.4) ---
POISSON = 0.30                    # Poisson's ratio, typical metal
geometric = 1 + 2 * POISSON      # 1 + 2*nu  -> purely geometric contribution
piezoresistive = GF - geometric  # remainder, from the strain-dependence of rho
print()
print(f"Geometric part  (1 + 2*nu) = {geometric:.2f}")
print(f"Piezoresistive remainder   = {piezoresistive:.2f}")

---
## 3 — Lead-wire resistance in a two-wire quarter bridge

The wires that carry the gage signal are conductors too, and $R = \rho L / A$ applies to them just as it does to the grid. In a **two-wire quarter bridge**, both lead wires sit in series with the gage inside the *same* bridge arm, so their **total** resistance $R_L$ adds directly to the gage resistance $R_g$. That single fact produces two distinct errors, and it is worth keeping them separate:

- **Desensitization — a span error.** The lead resistance dilutes the strain signal: only the fraction $R_g / (R_g + R_L)$ of the gage's resistance change reaches the bridge. The relative sensitivity loss is $-R_L / (R_g + R_L) \approx -R_L / R_g$, and it scales the entire reading.
- **Apparent strain — a zero error.** The bridge is nulled with $R_L$ already present, so a *static* lead resistance does not itself read as strain. But if $R_L$ *drifts* — from temperature, connector oxidation, or a flexing cable — that change $\Delta R_L$ looks exactly like a gage resistance change and reads as an apparent strain $\Delta R_L / (R_g\,GF)$. The table below reports the worst case, in which the full lead resistance drifts.

Even 1 Ω of lead resistance — roughly a 10 m round trip of 26 AWG copper — is enough to matter. The remedy is the **three-wire connection**, which routes one lead through each of two adjacent bridge arms so their resistance changes subtract; use it whenever the cable run is long or the environment is uncontrolled.

In [ ]:
def lead_wire_errors(R_lead_total, R_gage, gage_factor):
    """Lead-wire errors in a two-wire quarter bridge.

    Parameters
    ----------
    R_lead_total : float or np.ndarray
        Total resistance of the two lead wires in series (ohm).
    R_gage : float
        Nominal gage resistance (ohm).
    gage_factor : float
        Gage factor of the gage (dimensionless).

    Returns
    -------
    desens_pct : float or np.ndarray
        Sensitivity loss (span error) as a percentage:
        -R_L / (R_g + R_L) * 100.
    apparent_ue : float or np.ndarray
        Worst-case apparent strain in microstrain if the full lead
        resistance were to drift: -R_L / (R_g * GF) * 1e6.
    """
    desens_pct = -R_lead_total / (R_gage + R_lead_total) * 100.0
    apparent_ue = -R_lead_total / (R_gage * gage_factor) * 1e6
    return desens_pct, apparent_ue

R_GAGE = 350.0                                    # ohm  (preferred for long leads)
GF_LEAD = 2.07                                    # gage factor
R_lead_values = np.array([0.0, 0.5, 1.0, 2.0, 5.0])   # ohm, total lead resistance

desens_pct, apparent_ue = lead_wire_errors(R_lead_values, R_GAGE, GF_LEAD)

print(f"Two-wire quarter bridge:  R_g = {R_GAGE:.0f} ohm,  GF = {GF_LEAD}")
print(f"{'R_L (ohm)':>10}  {'Desensitization (%)':>20}  {'Apparent strain (ue)':>22}")
for RL, d, a in zip(R_lead_values, desens_pct, apparent_ue):
    print(f"{RL:>10.1f}  {d:>20.3f}  {a:>22.1f}")
print()
print("A three-wire connection cancels lead resistance -> use it whenever possible.")

In [ ]:
# Visualize how both errors grow with lead resistance.
RL_fine = np.linspace(0, 5, 200)                 # ohm
desens_fine, apparent_fine = lead_wire_errors(RL_fine, R_GAGE, GF_LEAD)

fig, ax1 = plt.subplots(figsize=(6.5, 4))
ax1.plot(RL_fine, apparent_fine, color="tab:blue", label="Apparent strain")
ax1.axhline(0, color="0.7", lw=0.8)
ax1.set_xlabel("Total lead resistance $R_L$ (Ω)")
ax1.set_ylabel("Worst-case apparent strain (µε)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(RL_fine, desens_fine, color="tab:red", ls="--", label="Desensitization")
ax2.set_ylabel("Sensitivity loss (%)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

ax1.set_title(
    f"Two-wire lead error vs. lead resistance "
    f"($R_g$ = {R_GAGE:.0f} Ω, GF = {GF_LEAD})"
)
fig.tight_layout()
plt.show()

---
## Where to take this next

- **Add a three-wire model.** Write a companion to `lead_wire_errors` for the three-wire quarter bridge and confirm numerically that a *matched* change in the two leads cancels between adjacent arms — the quantitative case for the wiring practice this chapter recommends.
- **Sweep real wire tables, not guesses.** Replace the hand-picked lead resistances with actual AWG resistance-per-metre data (say 22–30 AWG copper) and plot apparent strain versus cable length, so a reader can size leads against a target error budget.
- **Fold a real grid.** Model the geometry behind "Why the Grid Is Folded": given an active grid length, number of passes, line width, and foil thickness, compute the total conductor length and resistance, then find which folding patterns land on 120 Ω versus 350 Ω.